# Codelab: Pre-permissioning ADK Agent Identities on GCP Agent Engine

This notebook demonstrates how to how to provision an **Agent Identity** on **Google Cloud's Agent Engine** before deploying ADK agent code to the engine. It also demonstrates how to assign permissions  **before** deploying the agent code and how to deploy the code later specifying an engine id.


The notebook can be executed in your local ide or in GCP's Colab Enterprise. If you choose a cloud deployment, make sure you have the necessary network for your runtime. Use the setup_compute_network.sh if you need to create a VPC.  

## Conceptual Q&A: Questions this lab is focused on.

### Q1: Is it possible to specify an existing identity?
**Answer:** No. Google’s Agent Identity is designed to act as a cryptographically attested, SPIFFE-based identity that is intrinsically tied to the lifecycle and resource path of a specific agent. Because it is auto-provisioned by the platform (complete with a short-lived, auto-rotating X.509 certificate), you can't "bring your own" identity to the deployment.

### Q2: Is it possible to pre-create / pre-permission an identity?
**Answer:** **Yes, via an "empty shell" deployment** (see the [Google Cloud Agent Identity documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/agent-identity#create-agent-identity)). You cannot predict the SPIFFE ID entirely offline because the Agent Engine ID is a platform-assigned numeric identifier generated at resource creation. However, you can provision an empty Agent Engine instance (without any agent code) to allocate the identity beforehand. Once the identity is allocated, you can programmatically capture its SPIFFE ID, assign the necessary permissions, and then later deploy your real agent code to that pre-existing engine instance.

### Q3: Am I just approaching this problem wrong?
**Answer:** No, your approach is fine. Pre-permissioning using an empty shell deployment to separate identity provisioning from code deployment is an industry-standard secure-by-default enterprise agent pattern.

## Code Walkthrough: Pre-permissioning in GCP

Let's programmatically execute these steps using Python and the Google Cloud CLI.

### Step 0: Setup and Verify Dependencies

To keep your notebook codebase clean, simple, and standard, we isolate our dependencies in a local virtual environment (`.venv`) using `uv` (a fast Python package installer and resolver).

#### 1. Setup Virtual Environment (One-time Terminal Setup)
Run the following single-line command in your terminal at the root of this project directory. This command automatically configures `uv` to use system keyring authentication to authenticate with your corporate Artifact Registry seamlessly:

```bash
uv venv && source .venv/bin/activate && uv pip install --keyring-provider subprocess -r requirements.txt
```

#### 2. Select the Kernel in your IDE
Once the command finishes:
1. Click on the **Kernel selection dropdown** in the top-right corner of this editor.
2. Choose the newly created virtual environment interpreter (located at `.venv/bin/python`).

#### 3. Run the Cell Below
After selecting the `.venv` kernel, run the code cell below to verify that all dependencies are loaded correctly.

In [ ]:
# Verify your local virtual environment kernel is active and packages are loaded
import sys

try:
    import vertexai
    from google.cloud import secretmanager
    import google.adk
    
    print("======================================================================")
    print("✓ SUCCESS: Environment is configured correctly!")
    print("======================================================================")
    print("Active Interpreter:", sys.executable)
    print("Vertex AI SDK version:", vertexai.__version__)
    print("======================================================================")
except ImportError as e:
    print("======================================================================")
    print("✗ ERROR: Missing dependencies or wrong kernel selected!")
    print("======================================================================")
    print(f"Details: {e}")
    print("\nTroubleshooting Steps:")
    print("1. Ensure you ran the setup command in your terminal:")
    print("   uv venv && source .venv/bin/activate && uv pip install --keyring-provider subprocess -r requirements.txt")
    print("2. Ensure you selected the '.venv' kernel in the top-right corner of your IDE.")
    print("======================================================================")

### Step 1: Discover Project Details
Let's retrieve the current Project ID and Project Number programmatically.

In [ ]:
# Discover Google Cloud project details directly from your active gcloud session using shell commands (!)
PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0].strip()

# Retrieve project number using describe
PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
project_number = PROJECT_NUMBER[0].strip()

# Keep a lowercase version for backward compatibility in later cells
PROJECT_NUMBER = project_number

# Set environment variables so the Google SDKs automatically inherit the active project
import os
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['CLOUDSDK_CORE_PROJECT'] = PROJECT_ID

print(f"Project ID: {PROJECT_ID}")
print(f"Project Number: {project_number}")

### Step 2: Creating an "Empty Shell" Agent Engine (Identity-Only Provisioning)

In standard enterprise environments, you must establish all IAM permissions and security bindings **before** your developer team finishes writing or deploying the actual agent code. However, because Google's Agent Identity is strictly tied to a platform-assigned numeric resource ID that is dynamically allocated by the platform, **there is no way to predict the SPIFFE ID beforehand without first creating the Agent Engine resource.**

To solve this, we provision an **"empty shell"** — a lightweight, identity-only Agent Engine instance (for details, see the [Google Cloud Agent Identity documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/agent-identity#create-agent-identity)). By utilizing the Vertex AI `v1beta1` Python SDK client with `IdentityType.AGENT_IDENTITY` and omitting any actual agent code, we can allocate the resource, programmatically retrieve the resulting secure SPIFFE-based principal ID, and use it in IAM policy bindings before any agent logic is deployed.

In [ ]:
import vertexai
from vertexai import types

region = 'us-central1'

print("Initializing Vertex AI Client (v1beta1)...")
client = vertexai.Client(
    project=PROJECT_ID,
    location=region,
    http_options=dict(api_version="v1beta1")
)

print("Provisioning identity-only Agent Engine...")
remote_app = client.agent_engines.create(
    config={
        "display_name": "identity-only-setup",
        "identity_type": types.IdentityType.AGENT_IDENTITY,
    },
)

# Extract engine ID and Agent Identity details
engine_id = remote_app.api_resource.name.split('/')[-1]
effective_identity = remote_app.api_resource.spec.effective_identity
predicted_principal = f"principal://{effective_identity}"

print("======================================================================")
print("Agent Identity Created Successfully!")
print("======================================================================")
print(f"Agent Engine ID: {engine_id}")
print(f"SPIFFE ID:       {effective_identity}")
print(f"IAM Member ID:   {predicted_principal}")
print("======================================================================")
print("\nUse the following command to deploy your agent codebase to this identity:")
print(f"adk deploy agent_engine --project={PROJECT_ID} --region={region} --agent_engine_id={engine_id} secret_agent")
print("\nOr run the custom deployment script:")
print(f"./deploy_agent.sh {engine_id}")
print("======================================================================")

### Step 3: Create a Secret in Google Secret Manager
Let's create a secret representing the credentials needed to log into your internal tool.

In [ ]:
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient()
secret_id = "internal-tool-credentials"
parent = f"projects/{PROJECT_ID}"

try:
    secret = client.create_secret(
        request={
            "parent": parent,
            "secret_id": secret_id,
            "secret": {"replication": {"automatic": {}}},
        }
    )
    print(f"Secret created successfully: {secret.name}")
except Exception as e:
    if "AlreadyExists" in str(e) or "already exists" in str(e):
        print(f"Secret '{secret_id}' already exists.")
    else:
        raise e

version = client.add_secret_version(
    request={
        "parent": f"{parent}/secrets/{secret_id}",
        "payload": {"data": b"my-super-secret-internal-password"},
    }
)
print(f"Added secret version: {version.name}")

### Step 4: Pre-Permission the Identity on Secret Manager
Now we will grant the predicted principal ID the **Secret Manager Secret Accessor** role (`roles/secretmanager.secretAccessor`) on the specific secret.

In [ ]:
import subprocess

print(f"Granting secretAccessor role to predicted principal ID...")

cmd_iam = [
    'gcloud', 'secrets', 'add-iam-policy-binding', secret_id,
    f'--member={predicted_principal}',
    '--role=roles/secretmanager.secretAccessor',
    f'--project={PROJECT_ID}',
    '--format=json'
]

subprocess.check_call(cmd_iam)
print("IAM Policy updated successfully!")

### Step 5: Verify IAM Policy Binding
Let's retrieve the IAM policy of the secret to prove that the predicted Agent Identity principal is successfully permissioned.

In [ ]:
import json
import subprocess

iam_policy_json = subprocess.check_output([
    'gcloud', 'secrets', 'get-iam-policy', secret_id,
    f'--project={PROJECT_ID}',
    '--format=json'
]).decode('utf-8')

iam_policy = json.loads(iam_policy_json)
print("Current IAM Policy for Secret:")
print(json.dumps(iam_policy, indent=2))

### Step 6: Deploying the Agent Code to the Pre-provisioned Engine

Now that the identity has been created and successfully permissioned on the Secret Manager secret, we can deploy our actual agent code to the pre-provisioned "empty shell" engine.

To do this, we use the Google Agent Development Kit (ADK) CLI. When deploying your code:
* You can specify the ID of the engine using the `--agent_engine_id` flag.
* If you do **not** specify an engine ID, a brand-new reasoning engine and identity will be created.
* Because the engine was already created with the Agent Identity in Step 2, you do **not** have to specify or configure the identity again. The pre-provisioned resource already has the Agent Identity enabled.
* The deployment takes a couple of minutes.

We also have a shell version of this deployment command in [deploy_agent.sh](file:///Users/rodrigomm/source-agy/agent-identity/deploy_agent.sh) in this repository.

In [ ]:
import os
import subprocess
import sys

print(f"Deploying agent code to pre-provisioned Agent Engine ID: {engine_id}...")

# Find the ADK command line utility
adk_path = os.path.join(os.path.dirname(sys.executable), 'adk')
if not os.path.exists(adk_path):
    adk_path = 'adk'

cmd_deploy = [
    adk_path, 'deploy', 'agent_engine',
    f'--project={PROJECT_ID}',
    f'--region={region}',
    '--display_name=Secret Agent',
    '--description=Retrieves a secret from Google Cloud Secret Manager.',
    f'--agent_engine_id={engine_id}',
    'secret_agent'
]

try:
    subprocess.check_call(cmd_deploy)
    print("Agent deployed successfully!")
except Exception as e:
    print(f"Could not deploy agent programmatically: {e}")
    print(f"You can run this manually in your shell:\n\n{' '.join(cmd_deploy)}")

### Step 7: (Optional) Live Deployment Verification
If you have deployed a live agent (which has a system-assigned numeric ID), you can run this section to dynamically discover its active principal ID, grant it access, and verify the end-to-end integration!

In [ ]:
import json
import subprocess

try:
    print("Searching for live deployed 'Secret Agent' instances...")
    engines_res = subprocess.check_output([
        'gcloud', 'ai', 'reasoning-engines', 'list',
        f'--project={PROJECT_ID}',
        f'--region={region}',
        '--format=json'
    ]).decode('utf-8')
    engines = json.loads(engines_res)
    
    secret_agents = [e for e in engines if e.get('displayName') == 'Secret Agent']
    if secret_agents:
        live_resource_name = secret_agents[0]['name']
        live_id = live_resource_name.split('/')[-1]
        
        # Retrieve the effectiveIdentity directly from the API object
        effective_identity = secret_agents[0].get('spec', {}).get('effectiveIdentity')
        if effective_identity:
            live_principal = f"principal://{effective_identity}"
        else:
            # Fallback construct
            live_principal = f"principal://agents.global.org-placeholder-org.system.id.goog/resources/aiplatform/projects/{project_number}/locations/{region}/reasoningEngines/{live_id}"
            
        print(f"Found Live Deployed Agent Name: {live_resource_name}")
        print(f"Constructed Live Principal ID: {live_principal}")
        
        # Grant permission to the live principal
        print("Granting secretAccessor role to live principal...")
        cmd_iam_live = [
            'gcloud', 'secrets', 'add-iam-policy-binding', secret_id,
            f'--member={live_principal}',
            '--role=roles/secretmanager.secretAccessor',
            f'--project={PROJECT_ID}',
            '--format=json'
        ]
        subprocess.check_call(cmd_iam_live)
        print("Live Agent Policy updated successfully!")
    else:
        print("No live 'Secret Agent' deployment found. Skipping live authorization.")
except Exception as e:
    print(f"Skipping live verification: {str(e)}")

## Conclusion

By provisioning the Agent Identity beforehand without deploying any code (via an "empty shell" deployment), we successfully retrieved the actual SPIFFE-based principal ID and pre-permissioned our agent. The moment your deployment finishes, the agent's identity will instantly inherit the `secretAccessor` role and can retrieve the credentials without any access-denied window!